# 🤖 Random Forest — ทำนายผลพิจารณาคำขอกองทุนยุติธรรม

**Goal:** ใช้ข้อมูลจังหวัด ประเภทคำขอ จำนวนเงินที่ขอ เดือน ปี → ทำนายว่าคำขอจะได้ผลพิจารณาอย่างไร

**Model:** Random Forest Classifier

**Target:** `OpinionName` (อนุมัติ / ไม่อนุมัติ / ยุติสำนวน)

In [ ]:
# Google Colab — run this cell first
!pip install pandas scikit-learn plotly openpyxl -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_excel('66-68Stat_ServiceType.xlsx')
df['RequestAmount'] = pd.to_numeric(df['RequestAmount'], errors='coerce').fillna(0)
df['TotalAmount'] = pd.to_numeric(df['TotalAmount'], errors='coerce').fillna(0)

# Filter to top 3 outcomes for cleaner classification
top_opinions = ['อนุมัติ','ไม่อนุมัติ','ยุติสำนวน']
df_model = df[df['OpinionName'].isin(top_opinions)].copy()
print(f'Records for modeling: {len(df_model):,}')
print(df_model['OpinionName'].value_counts())

## Feature Engineering

In [ ]:
# Encode categorical features
le_prov = LabelEncoder()
le_case = LabelEncoder()
le_target = LabelEncoder()

df_model['prov_enc'] = le_prov.fit_transform(df_model['ProvinceName'])
df_model['case_enc'] = le_case.fit_transform(df_model['CaseTypeName'])
df_model['target'] = le_target.fit_transform(df_model['OpinionName'])

# Features
features = ['prov_enc','case_enc','Year','Month','CaseAmount','RequestAmount']
X = df_model[features].values
y = df_model['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Classes: {list(le_target.classes_)}')

## Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=5,
    random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n🎯 Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'\n📋 Classification Report:\n')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

## Cross-Validation

In [ ]:
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {[f"{s:.4f}" for s in cv_scores]}')

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(
    cm, text_auto=True,
    x=le_target.classes_, y=le_target.classes_,
    labels={'x':'Predicted','y':'Actual','color':'Count'},
    color_continuous_scale='YlOrRd',
    title='Confusion Matrix — Random Forest',
)
fig_cm.update_layout(
    font=dict(family='Sarabun, sans-serif'), height=450, width=550)
fig_cm.show()

## Feature Importance

In [ ]:
importances = rf.feature_importances_
feat_names = ['จังหวัด','ประเภทคำขอ','ปี','เดือน','จำนวนคำขอ','จำนวนเงินที่ขอ']
idx = np.argsort(importances)

fig_imp = go.Figure(go.Bar(
    x=importances[idx], y=[feat_names[i] for i in idx],
    orientation='h',
    marker=dict(color=importances[idx],
                colorscale=[[0,'#2a9d8f'],[1,'#e76f51']]),
    text=[f'{v:.3f}' for v in importances[idx]],
    textposition='outside',
))
fig_imp.update_layout(
    title='Feature Importance — Random Forest',
    xaxis_title='Importance', template='plotly_white',
    height=380, font=dict(family='Sarabun, sans-serif'),
    margin=dict(l=160),
)
fig_imp.show()

## สรุป

Random Forest Classifier สามารถทำนายผลพิจารณาคำขอกองทุนยุติธรรมได้จาก features ที่มี โดยมี accuracy ตามที่แสดงด้านบน

**Feature ที่สำคัญ** ได้แก่ จำนวนเงินที่ขอ และ ประเภทคำขอ ซึ่งสอดคล้องกับความเป็นจริงที่คำขอประเภทต่างๆ มีอัตราการอนุมัติที่แตกต่างกัน